In [1]:

# Load model directly
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

model_path = "/mnt/d/modeles/Qwen3.5-4B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

processor = AutoProcessor.from_pretrained(model_path)
model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    quantization_config=quantization_config,
    device_map="cuda",
)
print(f"Modèle chargé sur : {model.device}")


/mnt/c/Users/darkf/Desktop/Pixtrall/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


cuda


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d
Loading weights: 100%|██████████| 723/723 [01:17<00:00,  9.29it/s] 


Modèle chargé sur : cuda:0


In [ ]:

from transformers import TextStreamer

image = "./table_ronde.jpg"

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "url": image},
            {"type": "text", "text": "que vois tu sur l'image en francais"}
        ]
    },
]
inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
    enable_thinking=False,
).to(model.device)

streamer = TextStreamer(processor.tokenizer, skip_prompt=True, skip_special_tokens=True)

outputs = model.generate(**inputs, max_new_tokens=1024, streamer=streamer)


Setting `pad_token_id` to `eos_token_id`:248044 for open-end generation.


The user wants me to describe the image in French.

1.  **Identify the main subject:** It's a group of people sitting around a round table.
2.  **Describe the setting:** It looks like a meeting room or an office space. The floor is grey with white stripes (maybe sunlight or a design). There's a red carpet or rug in the bottom right corner.
3.  **Identify the people:** There are five people visible.
    *   Top left: A man in a light blue shirt and tie.
    *   Top right: A woman with reddish hair, looking down at papers.
    *   Right: A man in a blue shirt, looking at a laptop.
    *   Bottom left: A woman with blonde hair, looking at a laptop.
    *   Bottom right: A woman with brown hair, looking at a notepad/paper.
4.  **Describe the objects on the table:**
    *   Laptops (at least two visible).
    *   Papers/documents scattered around.
    *   Pens.
    *   A small white bowl or container.
    *   A notepad.
5.  **Describe the action:** They seem to be having a business meeting.